<a href="https://colab.research.google.com/github/tarabelo/GrIA-QML-2025-26/blob/main/05.%20Introducci%C3%B3n%20al%20Quantum%20Machine%20Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalamos qiskit en el notebook
!pip install qiskit[visualization] qiskit-aer qiskit-algorithms qiskit-optimization qiskit_machine_learning qiskit-ibm-runtime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.0/660.0 kB 12.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.8/381.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import numpy as np
from math import sqrt

# importing Qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator, StatevectorSimulator

# import basic plot tools
from qiskit.visualization import plot_histogram

# Funciones auxiliares

# Función para obtener y mostrar el vector de estado
def obten_estado(qcirc, etiqueta="|\psi\!\!> = ", bloch=False):
  estado = Statevector.from_instruction(qcirc)
  display(estado.draw('latex', prefix=etiqueta, max_size=2**qc.num_qubits))
  if bloch:
    display(estado.draw('bloch'))

<>:15: SyntaxWarning: invalid escape sequence '\p'
<>:15: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_260/420566059.py:15: SyntaxWarning: invalid escape sequence '\p'
  def obten_estado(qcirc, etiqueta="|\psi\!\!> = ", bloch=False):



# **Introducción al Quantum Machine Learning (QML)**

### Contenidos

1. [Representación de la información](#info)
1. [Problemas de optimización binaria cuadrática sin restricciones (QUBO)](#qubo)
1. [Computación cuántica adiabática y Quantum Annealing](#adiabatica)
1. [Algoritmos cuánticos híbridos y circuitos parametrizados](#hibrida)
1. [Quantum Approximate Optimization Algorithm (QAOA)](#qaoa)
1. [Variational Quantum Eigensolvers (VQE)](#vqe)
1. [Introducción al Quantum Machine Learning](#qml)
1. [Otras aplicaciones](#otras)

<a name="info"></a>
# **Representación de la información**

Codificar nuestros datos como un estado cuántico: diferentes soluciones propuestas

  - Problema abierto y bajo estudio
  - Dependiente del problema concreto

## Codificación en la base (Basis encoding)

La codificación más simple es usar los cúbits como bits clásicos. Así, por ejemplo, si tenemos 8 cúbits el valor $123$ se representaría como el estado $|01111011\rangle$

También se pueden agrupar los cúbits en _registros_, cada uno con un estado especificando un valor:

$$
\begin{bmatrix}
7\\11
\end{bmatrix} = |0111\rangle|1011\rangle
$$

- Ventajas: el estado es fácil de preparar
- Inconvenientes: necesitamos muchos cúbits

## Codificación en superposición (QuAM encoding)

Alternativamente, un vector de hasta $2^n$ enteros puede ser codificado en estados en superposición de $n$ cúbits.

Por ejemplo con 3 cúbits:

$$
\begin{bmatrix}
1\\3\\5\\6
\end{bmatrix} =
\frac{1}{2}(|001\rangle+|011\rangle + |101\rangle + |110\rangle)
$$

 - Ventajas: se pueden hacer operaciones que afectan a todos los elementos del vector simultáneamente.
 - Inconvenientes: la codificación no puede realizarse de forma eficiente

Un algoritmo para realizar esta codificación se basa en una [memoria cuántica asociativa (QuAM)](https://arxiv.org/abs/quant-ph/9807053)

Ejemplo: prepara el vector $[1,3,5,6]^T$

In [ ]:
q = QuantumRegister(3, name='q')
qc = QuantumCircuit(q)

## Iniciamos el estado
qc.x(q[0])
qc.h(q[1])
qc.h(q[2])
qc.ccx(q[2],q[1],q[0])

display(qc.draw('mpl'))

# Mostramos el vector de estado
obten_estado(qc)

Súmale 1 (módulo 8) a todos los elementos del vector

In [ ]:
q = QuantumRegister(3, name='q')
qc = QuantumCircuit(q)

## Iniciamos el estado
qc.x(q[0])
qc.h(q[1])
qc.h(q[2])
qc.ccx(q[2],q[1],q[0])
qc.barrier()

# Sumamos 1
qc.ccx(q[0],q[1],q[2])
qc.cx(q[0],q[1])
qc.x(q[0])

display(qc.draw('mpl'))

# Mostramos el vector de estado
obten_estado(qc)

In [ ]:
q = QuantumRegister(3, name='q')
qc = QuantumCircuit(q)

## Iniciamos el estado
qc.x(q[0])
qc.h(q[1])
qc.h(q[2])
qc.ccx(q[2],q[1],q[0])
qc.barrier()

# Sumamos 2
for i in range(2):
  qc.ccx(q[0],q[1],q[2])
  qc.cx(q[0],q[1])
  qc.x(q[0])

display(qc.draw('mpl'))

# Mostramos el vector de estado
obten_estado(qc)

## Codificación en amplitud

Un vector de $2^n$ elementos puede ser codificado en las amplitudes de $n$ cúbits.

Restricciones:

1. El número de elementos del vector debe ser potencia de 2
1. El vector debe estar normalizado
1. El proceso de codificación no es trivial (requiere tiempo exponencial en el número de qubits)

Uso práctico: necesidad de [QRAM](https://en.wikipedia.org/wiki/Quantum_memory) que permita cargar datos clásicos en registros cuánticos (no existe todavía).

**Ejemplo**: inicialización de un vector de $2^n$ elementos en un sistema fake de IBMQ:

In [ ]:
n = 5
# Creamos un vector aleatorio de 2**n elementos
v = np.random.random(2**n)
print(v)

In [ ]:
# Normalizamos v
vnorm = v / np.linalg.norm(v)
print(np.linalg.norm(vnorm))
print(vnorm)

In [ ]:
# Iniciamos un circuito de n cúbits al vector normalizado
qc = QuantumCircuit(n)
qc.initialize(vnorm)
display(qc.draw('mpl'))
obten_estado(qc, bloch=True)

In [ ]:
# Traspilamos a un FakeProvider
from qiskit_ibm_runtime.fake_provider import FakeOsaka
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

fake_backend = FakeOsaka()

# Creamos el Pass manager para ese backend
pass_manager = generate_preset_pass_manager(backend=fake_backend, optimization_level=1)

# Optimiza el circuito usando el pass manager
qc_optimizado = pass_manager.run(qc)

qc_optimizado.draw("mpl", idle_wires=False)

## Codificación en ángulos

Aplicamos a cada cúbit una rotación con un ángulo igual al valor a codificar

  - Necesitamos tantos cúbits como valores a codificar
  - Valores normalizados al intervalo $[-\pi,\pi]$
  - Fácil implementación

In [ ]:
n = 4 # Número de cúbits
v = np.random.random(n)
print(v)

qc = QuantumCircuit(n)
for q in range(n):
    qc.ry(v[q], q)

qc.draw('mpl')
obten_estado(qc, bloch=True)

Esta codificación puede complicarse haciendo uso de rotaciones adicionales para codificar $2n$ valores en $n$ cúbits (_dense angle encoding_)

In [ ]:
qc = QuantumCircuit(n/2)
for q in range(0,n,2):
    qc.ry(v[q], q//2)
    qc.p(v[q+1],q//2)

qc.draw('mpl')
obten_estado(qc, bloch=True)

Codificaciones de orden más alto (_higher order encoding_) pueden implicar puertas Hadamard, CNOTs entre los cúbits y rotaciones con ángulo producto

  - Pueden ser útiles en determinados problemas

In [ ]:
n = 2 # Número de cúbits
v = [-0.3, 2.5]

qc = QuantumCircuit(n)
qc.h(range(n))
for q in range(n):
    qc.ry(v[q], q)

qc.cx(0,1)
qc.rz(v[0]*v[1], 1)

display(qc.draw('mpl'))
obten_estado(qc, bloch=True)

## Codificación de matrices

Una matriz podría codificarse como un conjunto de vectores, pero la forma más habitual es codificarla como una matriz unitaria que se puede usar en algoritmos como el QPE

Sea $A$ una matriz hermítica ($A^\dagger = A$) de coeficientes complejos. Se puede demostrar que la matriz $U = e^{iA}$ es unitaria.

Además, si $U$ es unitaria, existe una matriz hermítica $A$ tal que $U = e^{iA}$.

Podemos codificar una matriz hermítica como una unitaria que se puede usar en nuestro circuito. Si nuestra matriz $A$ no es hermítica, siempre podemos convertirla en hermítica haciendo:

$$A_H=\begin{bmatrix}0 & A\\A^\dagger & 0\end{bmatrix}$$

#### Referencias

- Weigold, M., Barzen, J., Leymann, F., & Salm, M. (2021, March). [Expanding Data Encoding Patterns For Quantum Algorithms](https://www.iaas.uni-stuttgart.de/publications/Weigold2021_ExpandingDataEncodingPatterns.pdf). In 2021 IEEE 18th Int. Conf. on Software Architecture Companion (ICSA-C) (pp. 95-101).
- LaRose, R., & Coyle, B. (2020). [Robust data encodings for quantum classifiers](https://link.aps.org/pdf/10.1103/PhysRevA.102.032420). Physical Review A, 102(3), 032420.



---



---



---



<a name="qml"></a>
# **Introducción al Quantum Machine Learning (QML)**

El término Quantum Machine Learning (QML) se usa a menudo para referirse al más concreto quantum-enhanced machine learning: uso de algoritmos cuánticos (p.e. HHL) para acelerar la ejecución de problemas de ML.


<center><img src="https://drive.google.com/uc?export=view&id=1JZRB5zv-SBPZXxyG5oWpLt2q--13t_Oj" alt="Speedups de QML" width="1000"  /></center>
(Fuente: Biamonte, J., Wittek, P., Pancotti, N., Rebentrost, P., Wiebe, N., & Lloyd, S. (2017). Quantum machine learning. Nature, 549(7671), 195-202. <a href="https://www.nature.com/articles/nature23474">https://www.nature.com/articles/nature23474</a>)

<center><img src="https://drive.google.com/uc?export=view&id=1IoZw4HE4LRucwqVwZfNOr97ktIPraL9Z" alt="Speedups de QML" width="950"  /></center>
(Fuente: Amira Abbas, Building a quantum classifier, Qiskit Global Summer School 2021)


Realizar la computación del modelo en un sistema cuántico:

1. Codificamos los datos en un estado cuántico (pe. mediante codificación en ángulos)
2. Aplicamos un circuito variacional usando un conjunto de parámetros
3. Medimos un cierto observable y determinamos la clasificación en función de los resultados
4. Optimizamos los parámetros para resolver el problema


<center><img src="https://drive.google.com/uc?export=view&id=15DbNGFMLC1qmLMHyjlPoE9y12K7HI3Ew" alt="VQC" width="800"  /></center>
(Fuente: Bryce Fuller, Quantum Support Vector Machines, Qiskit Global Summer School 2021)

## Quantum Support Vector Machines

<center><img src="https://drive.google.com/uc?export=view&id=15Z8lqGOIpDCiAfUZRqMumMOXu-wAqcQM" alt="SVM clásico" width="800"  /></center>



### [Kernel trick](https://en.wikipedia.org/wiki/Kernel_trick)

Se usa una transformación no lineal (_feature map_) $\Phi(x)$ para mapear los datos desde el espacio
original a un nuevo espacio de más dimensiones (espacio de características) donde la superficie de decisión (hiperplano) se vuelva lineal.

El hiperplano en este espacio se puede escribir como:

$$\omega^T\Phi(x) +b =0$$

y la función de clasificación:

$$y = \mathrm{label}(x) = \mathrm{sign}(\omega^T\Phi(x) +b)$$


#### Forma dual

En vez de calcular el hiperplano, se puede resolver el siguiente problema para obtener los _multiplicadores de Lagrange_ $\alpha_i$:

$$
\max_\alpha C_D(\alpha) = \sum_{i\in T} \alpha_i - \frac{1}{2}\sum_{i,j\in T} y_i y_j\alpha_i\alpha_j\Phi(x_i)^T\Phi(x_j)
$$
donde $T$ es el conjunto de entrenamiento y los valores $\alpha_i\ge 0$ solo son no-nulos para los vectores soporte en $T$.

La función de clasificación se puede escribir ahora como:

$$\mathrm{label}(s) = \mathrm{sign}\left(\sum_{i\in V}\alpha_iy_iK(s,x_i) +b\right)$$

donde $V$ es el conjunto de vectores soporte, $K$ la _función kernel_:

$$
K(x_i,x_j) = K_{ij} = \Phi(x_i)^T\Phi(x_j)
$$

y $b$ se obtiene a partir de cualquier vector de soporte $x_k$:

$$
b = y_k - \sum_{i\in T} \alpha_i y_i K(x_i, x_k)
$$


Los valores de la función (o matrix) kernel $K$ proporcionan una medida de _similaridad_ entre puntos, y pueden obtenerse sin necesidad de computar los productos internos $\Phi(x_i)^T\Phi(x_j)$ a través de funciones que codifiquen de forma implícita el feature map.

Ejemplo: [Radial Basis Function Kernel](https://en.wikipedia.org/wiki/Radial_basis_function_kernel)

$$
K(x_i,x_j) = \exp\left(-\frac{\lVert x_i-x_j\rVert^2}{2\sigma²}\right)
$$


### Variational Quantum Classifier (VQC)

<center><img src="https://drive.google.com/uc?export=view&id=10kCs4HAuz-5oEisufPGajVtzPMysnTpB" alt="Variational Quantum Classifier (VQC)" width="500"  /></center>
(Fuente: Bryce Fuller, Quantum Support Vector Machines, Qiskit Global Summer School 2021)

Midiendo el valor esperado del observable Z obtenemos:

$$
f_\theta(x) = \langle\Phi(x)|W^\dagger_\theta ZW_\theta|\Phi(x)\rangle \in [-1,1]
$$

Para la función de clasificación se elige un umbral $b\in [-1,1]$ y se define:

$$
\text{label}(x) =  \left\lbrace
\begin{array}{ll}
+1 & \text{si } f_\theta(x) \ge b\\
-1 & \text{si } f_\theta(x) < b
\end{array}
\right.
$$

Se puede demostrar que este modelo es un clasificador lineal en el espacio de características $\Phi(x)$ y $W_\theta$ parametriza el hiperplano.

Limitación: $W_\theta$ limitado por la profundidad del circuito $\Rightarrow$ no se puede probar con todos los hiperplanos posibles $\Rightarrow$ es posible que no se encuentre la solución óptima.

### Quantum Kernel Estimator (QKE)

Usa el computador cuántico solo para estimar la matriz kernel $K(x_i,x_j)$.

<center><img src="https://drive.google.com/uc?export=view&id=1naiNKfs5wrUw65xwvJf4EOA4qUDM_r36" alt="Quantum Kernel Estimator (QKE)" width="450"  /></center>
(Fuente: Bryce Fuller, Quantum Support Vector Machines, Qiskit Global Summer School 2021)

Obtenemos la matriz kernel midiendo la probabilidad de obtener el estado $|0\rangle$:

$$
K(x_i,x_j) = \text{Pr}(|0\rangle) = |\langle0|U^\dagger_{x_j} U_{x_i}|0\rangle|^2 = |\langle\Phi(x_j)|\Phi(x_i)\rangle|^2
$$

donde $U_x$ es la matriz unitaria tal que $|\Phi(x)\rangle = U_x|0\rangle $.

Se ha demostrado que QKE solo proporciona ventaja frente a un sistema clásico si $\Phi(x)$ es suficientemente compleja y difícil de simular clásicamente.

Ejemplos:

  - Forrelation kernel (https://doi.org/10.1137/15M1050902, https://doi.org/10.1145/3406325.3451040): kernel cuántico difícil de estimar
  - DLOG kernel (https://arxiv.org/pdf/2105.03406): kernel cuántico que aprovecha la estructura de los datos



### Ejemplos de Quantum Kernels en Qiskit

Qiskit tiene implementados algunos kernels, descritos en https://www.nature.com/articles/s41586-019-0980-2:

  - [`PauliFeatureMap`](https://qiskit.org/documentation/stubs/qiskit.circuit.library.PauliFeatureMap.html)   
  - [`ZFeatureMap`](https://qiskit.org/documentation/stubs/qiskit.circuit.library.ZFeatureMap.html)
  - [`ZZFeatureMap`](https://qiskit.org/documentation/stubs/qiskit.circuit.library.ZZFeatureMap.html)
  
En concreto, el ZZFeatureMap esta considerado como difícil de simular en un sistema clásico.

In [ ]:
from qiskit.circuit.library import ZZFeatureMap
map_zz = ZZFeatureMap(feature_dimension=4, reps=1, entanglement='full', insert_barriers=True)
map_zz.decompose().draw('mpl')

#### Ejemplo de clasificación binaria

Ejemplo del [Qiskit Machine Learning Tutorial](https://qiskit.org/documentation/machine-learning/tutorials/03_quantum_kernel.html).

Este ejemplo usa el dataset descrito en https://arxiv.org/pdf/1804.11326.pdf y el algoritmo [SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC) (_Support Vector Machine Classification_) del módulo de [SVM](https://scikit-learn.org/stable/modules/svm.html) de la librería [scikit-learn](https://scikit-learn.org/stable/).

Este ejemplo usa un dataset a medida descrito en este [artículo](https://arxiv.org/pdf/1804.11326). Ver [aquí](https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.datasets.ad_hoc_data.html) para detalles.

Definimos la dimensión del dataset y obtenemos los conjuntos de entrenamiento y test:


In [ ]:
from qiskit_machine_learning.datasets import ad_hoc_data

adhoc_dimension = 2
# adhoc_total son todos los puntos en una rejilla uniforme de la que se seleccionan las muestras
train_features, train_labels, test_features, test_labels, adhoc_total = ad_hoc_data(
    training_size=20, # 20 datos de cada clase, 40 en total
    test_size=5,      # 5 datos de cada clase, 10 en total
    n=adhoc_dimension,
    gap=0.3,
    plot_data=False,
    one_hot=False,
    include_sample_total=True,
)

In [ ]:
print(adhoc_total[0])

El dataset es bidimensional. Las dos características corresponden a coordenadas $x$ e $y$, y tiene dos clases, con etiquetas A y B. Las siguientes funciones permiten visualizarlo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_features(ax, features, labels, class_label, marker, face, edge, label):
    # A train plot
    ax.scatter(
        # x coordinate of labels where class is class_label
        features[np.where(labels[:] == class_label), 0],
        # y coordinate of labels where class is class_label
        features[np.where(labels[:] == class_label), 1],
        marker=marker,
        facecolors=face,
        edgecolors=edge,
        label=label,
    )

def plot_dataset(train_features, train_labels, test_features, test_labels, adhoc_total):
    plt.figure(figsize=(5, 5))
    plt.ylim(0, 2 * np.pi)
    plt.xlim(0, 2 * np.pi)
    # Imprime la rejilla
    plt.imshow(
        np.asmatrix(adhoc_total).T,
        interpolation="nearest",
        origin="lower",
        cmap="RdBu",
        extent=[0, 2 * np.pi, 0, 2 * np.pi],
    )
    # Entrenamiento etiqueta A
    plot_features(plt, train_features, train_labels, 0, "s", "w", "b", "A entrenamiento")
    # Entrenamiento etiqueta B
    plot_features(plt, train_features, train_labels, 1, "o", "w", "r", "B entrenamiento")
    # Test etiqueta A
    plot_features(plt, test_features, test_labels, 0, "s", "b", "w", "A test")
    # Test etiqueta B
    plot_features(plt, test_features, test_labels, 1, "o", "r", "w", "B test")

    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.0)
    plt.title("Dataset ad hoc para clasificación")

    plt.show()

In [ ]:
plot_dataset(train_features, train_labels, test_features, test_labels, adhoc_total)

Creamos un QuantumKernel para la clasificación. Usaremos un [FidelityQuantumKernel](https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.kernels.FidelityQuantumKernel.html) y como feature_map un [ZZFeatureMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.ZZFeatureMap).


In [ ]:
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

# Definimos el ZZFeatureMap con 2 repeticiones
adhoc_feature_map = ZZFeatureMap(feature_dimension=adhoc_dimension, reps=2, entanglement="linear")

adhoc_kernel = FidelityQuantumKernel(feature_map=adhoc_feature_map)

#### Clasificación usando SVC

El algortimo SVC de scikit-learn acepta dos formas de definir un kernel a medida.

La primera es pasándole la función que se encarga de computar la matriz kernel, que es el método evaluate del FidelityQuantumKernel

In [ ]:
from sklearn.svm import SVC

# Le pasamos la función que se encarga de computar la matriz kernel
adhoc_svc = SVC(kernel=adhoc_kernel.evaluate)

# Le pasamos los datos de entrenamiento
adhoc_svc.fit(train_features, train_labels)

# Obtenemos la calificación media de la clasificación de los datos de test
adhoc_score_funcion = adhoc_svc.score(test_features, test_labels)


print(f'Calificación media con los datos de test: {adhoc_score_funcion}')


La segunda forma es pasándole la matriz kernel precalculada.

Precalculamos las matrices $K(x_i,x_j)$ y $K(t_i, x_j)$ donde las $x$ son los datos de entrenamiento y las $t$ los de test

In [ ]:
# Obtenemos la matriz kernel para los datos de entrenamiento
adhoc_matrix_train = adhoc_kernel.evaluate(x_vec=train_features)

# Obtenemos la matriz kernel para los datos de test
adhoc_matrix_test = adhoc_kernel.evaluate(x_vec=test_features,
                                          y_vec=train_features)
# Mostramos las matrices
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(np.asmatrix(adhoc_matrix_train),
              interpolation='nearest', origin='upper', cmap='Blues')
axs[0].set_title("Matriz kernel datos de entrenamiento")
axs[1].imshow(np.asmatrix(adhoc_matrix_test),
              interpolation='nearest', origin='upper', cmap='Reds')
axs[1].set_title("Matriz kernel datos de test")
plt.show()

In [ ]:
# Ejecuta el SVC con las matrices precalculadas
adhoc_svc = SVC(kernel='precomputed')

# Le pasamos la matriz kelnel de entrenamiento y las etiquetas
adhoc_svc.fit(adhoc_matrix_train, train_labels)

# Obtenemos la calificación media de la clasificación de los datos de test
adhoc_score_precomputado = adhoc_svc.score(adhoc_matrix_test, test_labels)

print(f'Calificación media de los datos de test: {adhoc_score_precomputado}')

#### Clasificación usando QSVC

[QSVC](https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.algorithms.QSVC.html#qsvc) es una función de conveniencia proporcionada por Qiskit.


In [ ]:
from qiskit_machine_learning.algorithms import QSVC

qsvc = QSVC(quantum_kernel=adhoc_kernel)

qsvc.fit(train_features, train_labels)

qsvc_score = qsvc.score(test_features, test_labels)

print(f"QSVC: Calificación media de los datos de test: {qsvc_score}")



---



## Quantum Neural Networks

<center><img src="https://drive.google.com/uc?export=view&id=1OHw-beJcEnrWXca_w7Cn6DydqweHJeCo" alt="QNN" width="900"  /></center>
(Fuente: Mangini, S. et al. (2021). Quantum computing models for artificial neural networks. EPL (Europhysics Letters), 134(1), 10002. <a href='https://doi.org/10.1209/0295-5075/134/10002'>https://doi.org/10.1209/0295-5075/134/10002</a>)

- Estructura similar, pero diferente flujo de información

Se han propuesto otros modelos de, por ejemplo, redes neuronales convolucionales cuánticas o redes neuronales tensoriales:

  - Li, Y., Zhou, R. G., Xu, R., Luo, J., & Hu, W. (2020). A quantum deep convolutional neural network for image recognition. Quantum Science and Technology, 5(4), 044003. http://www.doi.org/10.1088/2058-9565/ab9f93
  - Henderson, M., Shakya, S., Pradhan, S., & Cook, T. (2020). Quanvolutional neural networks: powering image recognition with quantum circuits. Quantum Machine Intelligence, 2(1), 1-9. http://www.doi.org/10.1007/s42484-020-00012-y
  - Grant, E., Benedetti, M., et al. (2018). Hierarchical quantum classifiers. npj Quantum Information, 4(1), 1-8 .https://doi.org/10.1038/s41534-018-0116-9
  

## Librerías de QML

Muchos algoritmos están implementados en librerías de alto nivel:

#### [Qiskit ML](https://qiskit-community.github.io/qiskit-machine-learning/)

- Algoritmos antes incluidos en [Qiskit Aqua](https://github.com/Qiskit/qiskit-aqua)
    - Qiskit Aqua separado en [_Optimization_](https://qiskit-community.github.io/qiskit-optimization/), [_Finance_](https://qiskit-community.github.io/qiskit-finance/), [_Machine Learning_](https://qiskit-community.github.io/qiskit-machine-learning/) y [_Nature_](https://qiskit-community.github.io/qiskit-nature/)
- Ejemplos:
    - [Quantum Kernel Machine Learning](https://qiskit.org/documentation/machine-learning/tutorials/03_quantum_kernel.html)
    - [Quantum Neural Networks](https://qiskit.org/documentation/machine-learning/tutorials/01_neural_networks.html)
    - [Neural Network Classifier & Regressor](https://qiskit.org/documentation/machine-learning/tutorials/02_neural_network_classifier_and_regressor.html)
    - [PyTorch qGAN (Quantum Generative Adversarial Network) Implementation](https://qiskit.org/ecosystem/machine-learning/tutorials/04_torch_qgan.html)
- Permite diseñar redes neuronales híbridas con PyTorch

<center><img src="https://drive.google.com/uc?export=view&id=1Pi-UZ5KMHsxIjU-Mj2bn26NkfIrFTsEh" alt="Red neuronal híbrida" width="800"  /></center>
    
#### [Pennylane](https://pennylane.ai/)

- Librería cross-platform para [programación diferenciable](https://en.wikipedia.org/wiki/Differentiable_programming) de computadores cuánticos
- Desarrollada por la empresa [Xanadu Quantum Technologies](https://www.xanadu.ai/)
- Integra librerías de ML con diferentes simuladores y hardware cuántico:
    - [IBM Qiskit](https://docs.pennylane.ai/projects/qiskit/), [Microsoft Q#](https://docs.pennylane.ai/projects/qsharp), [Cirq](https://docs.pennylane.ai/projects/cirq), [Amazon Braket](https://amazon-braket-pennylane-plugin-python.readthedocs.io/en/latest/), etc.
    - Más info: https://pennylane.ai/plugins.html
- Interfaces con [Numpy](https://pennylane.readthedocs.io/en/stable/introduction/interfaces/numpy.html), [TensorFlow](https://pennylane.readthedocs.io/en/stable/introduction/interfaces/tf.html), [PyTorch](https://pennylane.readthedocs.io/en/stable/introduction/interfaces/torch.html) y [JAX](https://pennylane.readthedocs.io/en/stable/introduction/interfaces/jax.html)
- Más información:
    - Documentación: https://pennylane.readthedocs.io/
    - Demos: https://pennylane.ai/qml/demonstrations.html
    
#### [TensorFlow Quantum](https://www.tensorflow.org/quantum)

- Framework Python para Quantum Machine Learning
- Modelos híbridos clásicos-cuánticos
- Diseñado para trabajar con [Google Circ](https://quantumai.google/cirq)

<center><img src="https://drive.google.com/uc?export=view&id=1qdmRxTe5PgzpqLaksqwv9EaaHdNcXwx8" alt="TensorFlow" width="800"  /></center>
(Fuente: <a href='https://ai.googleblog.com/2020/03/announcing-tensorflow-quantum-open.html'>https://ai.googleblog.com/2020/03/announcing-tensorflow-quantum-open.html</a>)

#### Referencias:

  - Peral-García, D., Cruz-Benito, J., & García-Peñalvo, F. J. (2024). Systematic literature review: Quantum machine learning and its applications. Computer Science Review, 51, 100619. https://doi.org/10.1016/j.cosrev.2024.100619
  - Jerbi, S., Fiderer, L. J., Poulsen Nautrup, H., Kübler, J. M., Briegel, H. J., & Dunjko, V. (2023). Quantum machine learning beyond kernel methods. Nature Communications, 14(1), 1-8. https://www.nature.com/articles/s41467-023-36159-y
  - Cerezo, M., Verdon, G., Huang, H. Y., Cincio, L., & Coles, P. J. (2022). Challenges and opportunities in quantum machine learning. Nature Computational Science, 2(9), 567-576. https://www.nature.com/articles/s43588-022-00311-3
  - Huang, H. Y., Broughton, M., Mohseni, M., Babbush, R., Boixo, S., Neven, H., & McClean, J. R. (2021). Power of data in quantum machine learning. Nature communications, 12(1), 2631. https://www.nature.com/articles/s41467-021-22539-9
  - Liu, Y., Arunachalam, S., & Temme, K. (2021). A rigorous and robust quantum speed-up in supervised machine learning. Nature Physics, 1-5. https://doi.org/10.1038/s41567-021-01287-z  
  - Beer, K., Bondarenko, D., Farrelly, T., Osborne, T. J., Salzmann, R., Scheiermann, D., & Wolf, R. (2020). Training deep quantum neural networks. Nature communications, 11(1), 1-6, https://www.nature.com/articles/s41467-020-14454-2
  - Havlíček, V., Córcoles, A. D., Temme, K., Harrow, A. W., Kandala, A., Chow, J. M., & Gambetta, J. M. (2019). Supervised learning with quantum-enhanced feature spaces. Nature, 567(7747), 209-212. https://doi.org/10.1038/s41586-019-0980-2 https://arxiv.org/pdf/1804.11326.pdf
  - Schuld, M., Sweke, R., & Meyer, J. J. (2021). Effect of data encoding on the expressive power of variational quantum-machine-learning models. Physical Review A, 103(3), 032430. https://doi.org/10.1103/PhysRevA.103.032430
    
Más referencias en https://quantumalgorithmzoo.org/



---



---



---



<a name="otras"></a>
# **Otras aplicaciones**

El uso de la computación cuántica se ha extendido a muchos otros campos

![Campos de uso](https://github.com/tarabelo/2024-VIU-Quantum/blob/main/images/ecosistema2.png?raw=1)
(Fuente: https://www.bcg.com/publications/2018/next-decade-quantum-computing-how-play, 2018)

### Finanzas

En el ámbito financiero es en el que se ha despertado un mayor interés por la computación cuántica como mecanismo de acelerar sus operaciones.

- Herman, D., Googin, C., Liu, X., Sun, Y., Galda, A., Safro, I., ... & Alexeev, Y. (2023). Quantum computing for finance. Nature Reviews Physics, 5(8), 450-465. https://www.nature.com/articles/s42254-023-00603-1
- Wilkens, S., & Moorhouse, J. (2023). Quantum computing for financial risk measurement. Quantum Information Processing, 22(1), 51. https://doi.org/10.1007/s11128-022-03777-2
- Naik, A., Yeniaras, E., Hellstern, G., Prasad, G., & Vishwakarma, S. K. L. P. (2023). From portfolio optimization to quantum blockchain and security: A systematic review of quantum computing in finance. arXiv preprint arXiv:2307.01155. https://arxiv.org/abs/2307.01155
- Egger, D. J., Gambella, C., Marecek, J., McFaddin, S., Mevissen, M., Raymond, R., ... & Yndurain, E. (2020). Quantum computing for finance: State-of-the-art and future prospects. IEEE Transactions on Quantum Engineering, 1, 1-24. https://doi.org/10.1109/TQE.2020.3030314
- Egger, D. J., Gutierrez, R. G., Mestre, J. C., & Woerner, S. (2020). Credit risk analysis using quantum computers. IEEE Transactions on Computers. https://doi.org/10.1109/TC.2020.3038063
- McKinsey & Company (2020) [_How quantum computing could change financial services](https://www.mckinsey.com/industries/financial-services/our-insights/how-quantum-computing-could-change-financial-services)
- IBM, [_Exploring quantum computing use cases for financial services_](https://www.ibm.com/thought-leadership/institute-business-value/report/exploring-quantum-financial)
- [Qiskit Finance Tutorials](https://qiskit.org/documentation/tutorials/finance/index.html)

### Procesamiento de imágenes y visión por computador

Existen diferentes mecanismos de representación de imágenes que permiten una codificación eficiente de una imagen clásica en un estado cuántico, por ejemlo _Flexible Representation of Quantum Images (FRQI)_ ([Le, P.Q., Dong, F. & Hirota, K, 2011](https://doi.org/10.1007/s11128-010-0177-y)) y _Novel Enhanced Quantum Representation (NEQR) for Digital Images_ ([Zhang, Y., Lu, K., Gao, Y. et al., 2013](https://doi.org/10.1007/s11128-013-0567-z))

**Otros trabajos**

- Yan, F., Venegas-Andraca, S. E., & Hirota, K. (2022). Toward implementing efficient image processing algorithms on quantum computers. Soft Computing, 1-13. https://doi.org/10.1007/s00500-021-06669-2
- Wang, Z., Xu, M., & Zhang, Y. (2022). Review of quantum image processing. Archives of Computational Methods in Engineering, 29(2), 737-761. https://doi.org/10.1007/s11831-021-09599-2
- Das, S., Zhang, J., Martina, S., Suter, D., & Caruso, F. (2023). Quantum pattern recognition on real quantum processing units. Quantum Machine Intelligence, 5(1), 16. https://doi.org/10.1007/s42484-022-00093-x
- Zhou, N. R., Liu, X. X., Chen, Y. L., & Du, N. S. (2021). Quantum K-Nearest-Neighbor Image Classification Algorithm Based on KL Transform. International Journal of Theoretical Physics, 1-16. https://doi.org/10.1007/s10773-021-04747-7

### Bioinformática y genética

- Chagneau, A., Massaoudi, Y., Derbali, I., & Yahiaoui, L. (2024). Quantum algorithm for bioinformatics to compute the similarity between proteins. IET Quantum Communication. https://ietresearch.onlinelibrary.wiley.com/doi/full/10.1049/qtc2.12098
- Mokhtari, M., Khoshbakht, S., Ziyaei, K., Akbari, M. E., & Moravveji, S. S. (2024). New classifications for quantum bioinformatics: Q-bioinformatics, QCt-bioinformatics, QCg-bioinformatics, and QCr-bioinformatics. Briefings in Bioinformatics, 25(2), bbae074. https://academic.oup.com/bib/article/25/2/bbae074/7621030
- Fedorov, A. K., & Gelfand, M. S. (2021). Towards practical applications in quantum computational biology. Nature Computational Science, 1(2), 114-119. https://www.nature.com/articles/s43588-021-00024-z
- Sarkar, A., Al-Ars, Z., Almudever, C. G., & Bertels, K. (2019). An algorithm for DNA read alignment on quantum accelerators. arXiv preprint [arXiv:1909.05563](https://arxiv.org/abs/1909.05563)
- Sarkar, A., Al-Ars, Z., & Bertels, K. (2021). QuASeR: Quantum Accelerated de novo DNA sequence reconstruction. Plos one, 16(4), e0249850 https://doi.org/10.1371/journal.pone.0249850
- Cordier, B. A., Sawaya, N. P., Guerreschi, G. G., & McWeeney, S. K. (2022). Biology and medicine in the landscape of quantum advantages. Journal of the Royal Society Interface, 19(196). https://doi.org/10.1098/rsif.2022.0541

### Robótica

- Atchade-Adelomou P., Alonso-Linaje, G., Albo-Canals, J., Casado-Fauli, D. (2021). qRobot: A Quantum computing approach in mobile robot order picking and batching problem solver optimization. arXiv preprint [arXiv:2105.04865](https://arxiv.org/abs/2105.04865)
- Mannone, M., Seidita, V., & Chella, A. (2023). Modeling and designing a robotic swarm: A quantum computing approach. Swarm and Evolutionary Computation, 79, 101297. https://doi.org/10.1016/j.swevo.2023.101297
- Chella, A., Gaglio, S., Mannone, M., Pilato, G., Seidita, V., Vella, F., & Zammuto, S. (2023). Quantum planning for swarm robotics. Robotics and Autonomous Systems, 161, 104362. https://doi.org/10.1016/j.robot.2023.104362


## Algunas empresas de tecnologías cuánticas

<center><img src="https://drive.google.com/uc?export=view&id=1YWunTU_o6aM6cBjII2WnOAgo12j_fZTi" alt="Ecosistema empresarial" width="800"  /></center>

(Fuente: https://www.bcg.com/publications/2018/next-decade-quantum-computing-how-play, 2018)

